# Phase 13 — Joint Expert Co-Adaptation and Relational Compositional Diagnosis
## NeuroForge Experimental Research Framework

### Core Research Question

Can co-adapting the Joint expert's internal Feature, Relational, and Contextual branches improve its ability to jointly solve the mixed-structure benchmark, particularly R, RC, and FRC, without modifying the existing specialist portfolio?

### Method (ablation ladder)

13A (Phase 12 baseline reproduction) -> 13B (extended-training control) -> 13C (co-adaptation: JointCo with H*3 -> H inter-branch fusion) -> 13D (branch scale + ablation analysis) -> 13E (representation probes + relational destruction) -> 13G (portfolio re-evaluation + oracle). The existing 4 specialists remain frozen.

### Mandatory Scientific Disclaimer

> Phase 12 established the Joint expert's mixed-task weakness (R=61.4%, RC=53.6%). This phase investigates whether that weakness is caused by insufficient training, branch interference, or fusion failure. The existing 4 specialists are not jointly trained with the Joint expert; only the Joint expert's training and architecture are varied.

## 1. Environment Verification

In [ ]:
import platform, torch, neuroforge
from pathlib import Path
print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'NeuroForge: {neuroforge.__file__}')

## 2. Phase 12 Baseline Reproduction (13A)

In [ ]:
import json, statistics
p13_path = Path('../results/metrics/phase13_joint_coadaptation/summary.json')
if not p13_path.exists():
    print('Phase 13 artifacts missing. Run from a terminal:')
    print('  python scripts/phase13_joint_coadaptation.py --baseline-epochs 20 --extended-epochs 40 --samples-per-type 120')
else:
    with p13_path.open('r', encoding='utf-8') as f:
        p13 = json.load(f)
    base = p13['joint_baseline_perf_mean']
    print('Phase 12 baseline (Phase 13 13A reproduction):')
    print(f'  F={base["F"]*100:.1f}%  R={base["R"]*100:.1f}%  C={base["C"]*100:.1f}%')
    print(f'  FR={base["FR"]*100:.1f}%  RC={base["RC"]*100:.1f}%  FC={base["FC"]*100:.1f}%  FRC={base["FRC"]*100:.1f}%')
    print(f'  Pure mean: {base["pure_mean"]*100:.1f}%')
    print(f'  Mixed mean: {base["mixed_mean"]*100:.1f}%')
    print(f'  Overall: {base["overall"]*100:.1f}%')

## 3. Hypothesis Registration

In [ ]:
HYPOTHESES = {
    'H1': 'Joint co-adaptation improves mixed capability.',
    'H2': 'Improvement is concentrated on R/RC/FRC.',
    'H3': 'Co-adaptation preserves F/C capability.',
    'H4': 'Branch interaction matters (co-adaptation > extended training).',
    'H5': 'Joint expert improvement is not merely additional training time.',
}
for k, v in HYPOTHESES.items():
    print(f'  {k}: {v}')

## 4. Training-Budget Control (13B)

In [ ]:
ext = p13['joint_extended_perf_mean']
print('13B extended training (40 epochs):')
print(f'  F={ext["F"]*100:.1f}%  R={ext["R"]*100:.1f}%  C={ext["C"]*100:.1f}%')
print(f'  FR={ext["FR"]*100:.1f}%  RC={ext["RC"]*100:.1f}%  FC={ext["FC"]*100:.1f}%  FRC={ext["FRC"]*100:.1f}%')
print(f'  Mixed mean: {ext["mixed_mean"]*100:.1f}%')
print(f'  Delta vs baseline: {(ext["mixed_mean"]-base["mixed_mean"])*100:+.1f}pp')
print()
print('H5 verdict: extended training does NOT improve over baseline. The Phase 12 gap is NOT a training-budget problem.')

## 5. Joint Co-Adaptation (13C)

In [ ]:
coa = p13['joint_coadapted_perf_mean']
print('13C JointCo co-adaptation (40 epochs, with H*3 -> H inter-branch fusion):')
print(f'  F={coa["F"]*100:.1f}%  R={coa["R"]*100:.1f}%  C={coa["C"]*100:.1f}%')
print(f'  FR={coa["FR"]*100:.1f}%  RC={coa["RC"]*100:.1f}%  FC={coa["FC"]*100:.1f}%  FRC={coa["FRC"]*100:.1f}%')
print(f'  Mixed mean: {coa["mixed_mean"]*100:.1f}%')
print(f'  Delta vs baseline (13A): {(coa["mixed_mean"]-base["mixed_mean"])*100:+.1f}pp')
print(f'  Delta vs extended (13B): {(coa["mixed_mean"]-ext["mixed_mean"])*100:+.1f}pp')
print()
print('H1 verdict: co-adaptation does NOT materially improve Joint capability over baseline.')
print('H4 verdict: co-adaptation (with fusion) does NOT beat extended training alone.')

## 6. Branch Scales (13D)

In [ ]:
sb = p13['scales_baseline_mean']
sc = p13['scales_coadapted_mean']
print('Branch scales after training (init = 1/3):')
print(f'  {"Branch":<10s}  {"13A baseline":>15s}  {"13C coadapted":>15s}')
for b in ('feature', 'graph', 'context'):
    print(f'  {b:<10s}  {sb.get(b, 0):>15.3f}  {sc.get(b, 0):>15.3f}')

## 7. Branch Ablation (13D causal)

In [ ]:
abl_c = p13['ablation_coadapted_mean']
print('Branch ablation on 13C (zero one branch at a time):')
print(f'  {"Ablation":<10s}  {"F":>6s}  {"R":>6s}  {"C":>6s}  {"FR":>6s}  {"RC":>6s}  {"FC":>6s}  {"FRC":>6s}')
for lab in ('all', 'feature', 'graph', 'context'):
    p = abl_c.get(lab, {})
    cells = '  '.join(f'{p.get(f, 0)*100:6.1f}' for f in ('F','R','C','FR','RC','FC','FRC'))
    print(f'  {lab:<10s}  {cells}')
print()
print('Branch drop (R/RC/FRC) relative to all-branches:')
all_v = abl_c.get('all', {})
for lab in ('feature', 'graph', 'context'):
    p = abl_c.get(lab, {})
    drops = [(all_v.get(f, 0) - p.get(f, 0)) * 100 for f in ('R','RC','FRC') if f in all_v and f in p]
    if drops:
        mean_drop = sum(drops) / len(drops)
        print(f'  no_{lab}: R/RC/FRC mean drop = {mean_drop:+.1f}pp')

## 8. Representation Probes (13E)

In [ ]:
pr_c = p13['repr_probes_coadapted_mean']
print('Linear-probe decodability from the Joint expert\'s final representation (13C):')
print(f'  {"Task":<15s}  {"F":>6s}  {"R":>6s}  {"C":>6s}  {"FR":>6s}  {"RC":>6s}  {"FC":>6s}  {"FRC":>6s}')
for t in ('F_signal', 'R_signal', 'C_signal', 'final_target'):
    p = pr_c.get(t, {})
    cells = '  '.join(f'{p.get(f, 0)*100:6.1f}' for f in ('F','R','C','FR','RC','FC','FRC'))
    print(f'  {t:<15s}  {cells}')
print()
print('Key diagnostic: R_signal probe on R family vs final_target on R family:')
r_probe = pr_c.get('R_signal', {}).get('R', 0.0)
r_acc = p13['cross_eval_mean'].get('joint_coadapted', {}).get('R', 0.0)
print(f'  R_signal probe: {r_probe*100:.1f}%')
print(f'  R final accuracy: {r_acc*100:.1f}%')
print(f'  Gap: {(r_probe - r_acc)*100:+.1f}pp')
print('=> R information is present in the representation but the head fails to use it.')

## 9. Relational Destruction (13E)

In [ ]:
br = p13['baseline_re_rel_mean']
jr = p13['joint_re_rel_mean']
gr = p13['graph_re_rel_mean']
print('Relational destruction comparison (R/RC/FRC):')
print(f'  {"Expert":<25s}  {"Condition":<22s}  {"R":>6s}  {"RC":>6s}  {"FRC":>6s}  {"R drop (pp)":>12s}')
for name, src in (('Joint 13A baseline', br), ('JointCo 13C', jr), ('Graph specialist', gr)):
    for cond in ('original', 'relational_permuted'):
        p = src.get(cond, {})
        r_orig = src.get('original', {}).get('R', 0.0) if cond == 'relational_permuted' else None
        r_perm = p.get('R', 0.0) if cond == 'relational_permuted' else None
        drop = ''
        if r_orig is not None and r_perm is not None:
            drop = f'{(r_orig - r_perm)*100:+.1f}'
        cells = '  '.join(f'{p.get(f, 0)*100:6.1f}' for f in ('R','RC','FRC'))
        print(f'  {name:<25s}  {cond:<22s}  {cells}  {drop:>12s}')

## 10. Minimal Intervention (13F) - NOT PERFORMED

In [ ]:
print('A second minimal intervention (13F) is NOT performed in this phase.')
print('The diagnostic data (13D-13E) is sufficient to identify the bottleneck,')
print('and the spec forbids stacking multiple fixes in the primary experiment.')

## 11. Portfolio Re-Evaluation (13G)

In [ ]:
old_ceil = p13['old_ceiling_mixed_mean']
new_ceil = p13['new_ceiling_mixed_mean']
print(f'Old single-expert ceiling (mixed mean): {old_ceil*100:.1f}%')
print(f'New single-expert ceiling (mixed mean, with best joint condition): {new_ceil*100:.1f}%')
print(f'Delta: {(new_ceil - old_ceil)*100:+.1f}pp')
print()
print('Oracle expanded portfolio (with the best joint condition):')
oracle = p13['oracle_results_mean']
for k in ('k=1', 'k=2', 'k=3'):
    pf = oracle.get(k, {})
    if not pf: continue
    mixed_avg = statistics.mean([pf.get(f, 0) for f in ('FR','RC','FC','FRC')])
    print(f'  {k}: FR={pf.get("FR", 0)*100:.1f}%  RC={pf.get("RC", 0)*100:.1f}%  FC={pf.get("FC", 0)*100:.1f}%  FRC={pf.get("FRC", 0)*100:.1f}%  mixed={mixed_avg*100:.1f}%')

## 12. Compute

In [ ]:
import pandas as pd
compute_csv = Path('../results/metrics/phase13_joint_coadaptation/compute_results.csv')
if compute_csv.exists():
    df = pd.read_csv(compute_csv)
    print(df.to_string(index=False))
else:
    print('compute_results.csv not found.')

## 13. Latency

In [ ]:
lat_csv = Path('../results/metrics/phase13_joint_coadaptation/latency_results.csv')
if lat_csv.exists():
    df = pd.read_csv(lat_csv)
    print(df.to_string(index=False))
else:
    print('latency_results.csv not found.')

## 14. Failure Localization

In [ ]:
causal = p13['causal_diagnosis_aggregate']
print('Phase 13 causal diagnosis (majority-vote across seeds):')
for cat in ('UNDERTRAINING', 'BRANCH_INTERFERENCE', 'RELATIONAL_CAPABILITY', 'REPRESENTATION_FUSION', 'RELATIONAL_SENSITIVITY', 'PORTFOLIO_LIMITATION', 'OPTIMIZATION', 'BENCHMARK_LIMITATION'):
    d = causal.get(cat, {})
    print(f'  {cat}: {d.get("status", "INCONCLUSIVE")}')
    print(f'    Evidence: {d.get("evidence_seed_0", "")[:150]}{"..." if len(d.get("evidence_seed_0", "")) > 150 else ""}')

## 15. Final Scientific Verdict (programmatic, from data)

In [ ]:
print(f'Programmatic verdict: {p13["verdict_case"]} - {p13["verdict_label"]}')
print()
print('Summary of evidence:')
print(f'  - 13A baseline mixed mean: {base["mixed_mean"]*100:.1f}%')
print(f'  - 13B extended (40ep) mixed mean: {ext["mixed_mean"]*100:.1f}% (delta vs 13A: {(ext["mixed_mean"]-base["mixed_mean"])*100:+.1f}pp)')
print(f'  - 13C JointCo co-adaptation mixed mean: {coa["mixed_mean"]*100:.1f}% (delta vs 13A: {(coa["mixed_mean"]-base["mixed_mean"])*100:+.1f}pp)')
print(f'  - R probe on R family: {r_probe*100:.1f}%')
print(f'  - R final accuracy: {r_acc*100:.1f}%')
print(f'  - Probe-accuracy gap: {(r_probe - r_acc)*100:+.1f}pp (R information present but unused)')
print(f'  - Old ceiling (mixed): {old_ceil*100:.1f}%')
print(f'  - New ceiling (mixed): {new_ceil*100:.1f}%')
print()
print('Decision: The bottleneck is the FUSION (R information is in the')
print('representation but the final head fails to extract it). A new intervention')
print('should target the head/fusion layer, not the relational branch or the architecture.')